# Agent Basics

Core grasp-agents building blocks: typed agents, validated &
structured outputs, multimodal input, agentic tool loop, streaming, parallel
runs with retries/rate-limiting, sequential workflows, and agents-as-tools.

- For multi-agent orchestration, dynamic routing, and crash/resume durability,
  see `orchestration_durability_demo.ipynb`.
- For multimodal/provider-specific features and the full hook system, see
  `advanced_patterns_demo.ipynb`.

In [ ]:
import os
from typing import Any
from pydantic import Field, BaseModel
import httpx

from grasp_agents import (
    BaseTool,
    LLMAgent,
    RunContext,
    InputImage,
    print_events,
    render_events,
    ParallelProcessor,
)
from grasp_agents.types.events import ProcPacketOutEvent
from grasp_agents.llm_providers.openai_completions.completions_llm import OpenAILLM
from grasp_agents.llm_providers.litellm import LiteLLM, LiteLLMSettings
from grasp_agents.workflow.sequential_workflow import SequentialWorkflow
from grasp_agents.llm.cloud_llm import APIProvider
from grasp_agents.rate_limiting import RateLimiter
from grasp_agents.printer import Printer


from grasp_agents.telemetry import init_tracing

# Optional: enable LLM observability with Phoenix
from grasp_agents.telemetry.phoenix import init_phoenix

In [ ]:
# (no local assets - the image cells below fetch from URLs)

Images used in the demo

In [ ]:
IMG_1_URL = "https://www.simplilearn.com/ice9/free_resources_article_thumb/Expressions_In_C_2.PNG"
IMG_2_URL = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"

**[Optional] Observability with Arize Phoenix (deployed locally via Docker)**

Start Phoenix locally:
```bash
cd ./phoenix
docker compose up -d  
docker compose logs -f phoenix
export PHOENIX_COLLECTOR_HTTP_ENDPOINT=http://localhost:6006/v1/traces # from docker-compose.yml
export TELEMETRY_PROJECT_NAME_KEY="openinference.project.name" # required by Phoenix
```

Open `http://localhost:6006/` (or the port you set in `docker-compose.yml`) in your browser to access the Phoenix UI.

Initialize telemetry

In [ ]:
# Use Traceloop to produce spans (without sending them anywhere yet)
# init_tracing(project_name="agents-demo")

# # Use Phoenix as the backend and UI for telemetry
# # Alternatively, any OpenTelemetry-compatible backend can be used instead
# init_phoenix(batch=False, use_litellm_instr=True, use_llm_provider_instr=False)

## Simple generation with validated outputs

Output type validation

In [ ]:
# list[int] is the output type used to validate the output
from grasp_agents.llm import RetryPolicy

# A RunContext owns the session (usage tracker, stores, an optional printer).
# These result-focused cells just print the typed return value; to *watch* a
# run, wrap it in render_events(...) (the Console) as the streaming cells below do.
ctx = RunContext[None](printer=Printer())

chatbot = LLMAgent[None, list[int], None](
    name="chatbot",
    ctx=ctx,
    llm=LiteLLM(
        model_name="gpt-5.4-nano",
        retry_policy=RetryPolicy(validation_retries=2),
    ),
)

In [ ]:
# Code block delimiters are stripped from the output
out = await chatbot.run(
    "Output a list of 3 integers from 0 to 10 as a python array, no talking",
)
print(out.payloads[0])

In [ ]:
ctx.usage_tracker.usages

Streaming

In [ ]:
ctx = RunContext[None]()

chatbot = LLMAgent[None, list[int], None](
    name="chatbot",
    ctx=ctx,
    llm=LiteLLM(
        model_name="claude-sonnet-4-6",
        llm_settings=LiteLLMSettings(reasoning_effort=None),
    ),
    stream_llm=True,
)

In [ ]:
async for event in render_events(
    chatbot.run_stream(
        "Output a list of 30 integers from 0 to 10 as a python array. "
        "No code or talking.",
    )
):
    if isinstance(event, ProcPacketOutEvent):
        out = event.data

In [ ]:
ctx.usage_tracker.usages

## Raw debugging with the Printer

`render_events` (the Console) is for **readable** output. For **raw debugging**,
`print_events` shows everything a model produces or ingests — thinking,
response text, and tool-call arguments — streamed exactly, with no formatting.
The example below streams a reasoning model that calls a tool, so you can see
the thinking, the tool-call arguments, the tool result, and the response.

In [ ]:
# `print_events` is the *raw* counterpart to the Console (`render_events`):
# instead of pretty panels, it prints exactly what the model produces or ingests
# — thinking, response text, and tool-call arguments — streamed token-by-token
# and wrapped in <thinking>/<response>/<tool call>/<tool result> tags. Reach for
# it to debug precisely what a model emits; use the Console for readable output.


class MultiplyArgs(BaseModel):
    a: int
    b: int


class MultiplyTool(BaseTool[MultiplyArgs, int, Any]):
    name: str = "multiply"
    description: str = "Multiply two integers."

    async def _run(self, inp: MultiplyArgs, **kwargs: Any) -> int:
        return inp.a * inp.b


calc = LLMAgent[None, str, None](
    name="calc",
    llm=LiteLLM(
        model_name="claude-sonnet-4-6",
        llm_settings=LiteLLMSettings(reasoning_effort="low"),
    ),
    tools=[MultiplyTool()],
    sys_prompt="Use the multiply tool to compute products. Think briefly first.",
    stream_llm=True,
)

# Streams the thinking, the tool call (its arguments stream in), the tool
# result, and the final response — all raw, exactly as produced.
async for _ in print_events(calc.run_stream("What is 347 times 829?")):
    pass

Output type validation with structured outputs

In [ ]:
from enum import StrEnum


class Selector(StrEnum):
    A = "A"
    B = "B"


class Response(BaseModel):
    result: list[int] = Field(..., description="3 random integers")
    value: Selector = Field(..., description="Choose a value randomly")


ctx = RunContext[None](printer=Printer())

chatbot = LLMAgent[None, Response, None](
    name="chatbot",
    ctx=ctx,
    llm_output_schema=Response,
    llm=LiteLLM(
        model_name="gemini/gemini-2.5-flash",
        llm_settings=LiteLLMSettings(),
        # use provider-side output schema validation when available
        apply_output_schema_via_provider=True
    ),
)

# By default, llm_output_schema is set to the output type of the agent (Response) automatically.
# In some cases, you may want to set it to a different type, e.g. when using
# custom output parsing.

In [ ]:
out = await chatbot.run("start")

# Chat with images

In [ ]:
ctx = RunContext[None](printer=Printer())

chatbot = LLMAgent[None, str, None](
    name="chatbot",
    ctx=ctx,
    llm=LiteLLM(
        model_name="gemini/gemini-2.5-flash",
        llm_settings=LiteLLMSettings(reasoning_effort="disable"),
    ),
    sys_prompt="You are a helpful assistant",
)

In [ ]:
out = await chatbot.run("Where are you headed, stranger?")

In [ ]:
out = await chatbot.run("What did you just say, exactly?")

In [ ]:
out = await chatbot.run(
    ["What's in this image?", InputImage.from_url(IMG_1_URL)]
)

In [ ]:
out = await chatbot.run("Do the calculation")

In [ ]:
out = await chatbot.run(["Try another one", InputImage.from_url(IMG_2_URL)])

In [ ]:
out = await chatbot.run("What was my first question, exactly?")

In [ ]:
ctx.usage_tracker.total_usage

# Parallel runs with retries and rate limiting

In [ ]:
# Make the LLM generate text instead of integers occasionally
# to emphasise the need for retries

sys_prompt = """
You are a bad math student who always adds number {added_num} to the correct result of the operation. 
Output a single integer or its name, e.g. 'three' or '3'.
"""

in_prompt = "What is the square of {num}?"


class RunArgs(BaseModel):
    added_num: int


class InputArgs(BaseModel):
    num: int


# Specifying int as the output type means that the agent will
# validate the output against this type.

student = LLMAgent[InputArgs, int, RunArgs](
    name="student",
    llm=LiteLLM(
        model_name="gemini/gemini-3.1-flash-lite",
        llm_settings=LiteLLMSettings(temperature=2.0),
        # This rate limit will be applied to parallel runs of the agent
        rate_limiter=RateLimiter(rpm=100),
    ),
    sys_prompt=sys_prompt,
    in_prompt=in_prompt,
    reset_transcript_on_run=True,
    max_retries=2,
)


@student.add_system_prompt_builder
def system_prompt_builder(*, exec_id: str, **kwargs: Any) -> str:
    ctx = student.ctx
    return student.sys_prompt.format(added_num=ctx.state.added_num)

In [ ]:
in_args = [InputArgs(num=i) for i in range(10)]

In [ ]:
ctx = RunContext[RunArgs](state=RunArgs(added_num=5), printer=Printer())
out = await ParallelProcessor(student, ctx=ctx).run(in_args=in_args)

print()
print(*[p for p in out.payloads], sep="\n")

# Reasoning agent loop with streaming

In [ ]:
sys_prompt = """
Your task is to suggest an exciting stats problem to the student. 
You should first ask the student about their education, interests, and preferences, then suggest a problem tailored specifically to them. 

# Instructions
* Use the provided tool to ask questions.
* Ask questions one by one.
* The problem must have all the necessary data.
* Use the final answer tool to provide the problem.
"""

In [ ]:
# Tool input must be a Pydantic model to infer the JSON schema used by the LLM APIs
class TeacherQuestion(BaseModel):
    question: str


StudentReply = str


ask_student_tool_description = """
Ask the student a question and get their reply.

Args:
    question: str
        The question to ask the student.
Returns:
    reply: str
        The student's reply to the question.
"""


class AskStudentTool(BaseTool[TeacherQuestion, StudentReply, Any]):
    name: str = "ask_student"
    description: str = ask_student_tool_description

    async def _run(self, inp: TeacherQuestion, **kwargs: Any) -> StudentReply:
        return input(inp.question)

In [ ]:
class Problem(BaseModel):
    problem: str


teacher = LLMAgent[None, Problem, None](
    name="teacher",
    llm=LiteLLM(
        model_name="claude-sonnet-4-6",
        llm_settings=LiteLLMSettings(reasoning_effort="low"),
    ),
    tools=[AskStudentTool()],
    final_answer_as_tool_call=True,
    sys_prompt=sys_prompt,
    stream_llm=True,
)

In [ ]:
events = []
problem: Problem
async for event in render_events(teacher.run_stream("start")):
    if isinstance(event, ProcPacketOutEvent):
        problem = event.data.payloads[0]
    events.append(event)

In [ ]:
print(problem.problem)

# Sequential workflow 

In [ ]:
# Input arguments are passed to the agent dynamically (e.g. by other agents)
from grasp_agents.types.content import Content


# Global state is used to store data that is shared between runs of the agent.
class State(BaseModel):
    b: int
    c: int


class AddInputArgs(BaseModel):
    a: int = Field(..., description="First number to add.")


class AddResponse(BaseModel):
    a_plus_b: int


add_in_prompt = "Add {a} and {b}. Your only output is the resulting number."


add_agent = LLMAgent[AddInputArgs, AddResponse, State](
    name="add_agent",
    llm=LiteLLM(model_name="gpt-5.4-nano"),
    in_prompt=add_in_prompt,
    # Reset message history to system prompt (if provided) before each run
    reset_transcript_on_run=True,
    stream_llm=True,
)


@add_agent.add_input_content_builder
def build_input_content_impl(in_args: AddInputArgs, *, exec_id: str) -> Content:
    ctx = add_agent.ctx
    return Content.from_formatted_prompt(
        add_agent.in_prompt, a=in_args.a, b=ctx.state.b
    )


@add_agent.add_output_parser
def parse_output_impl(
    final_answer: str, *, in_args: AddInputArgs | None = None, exec_id: str
) -> AddResponse:
    return AddResponse(a_plus_b=int(final_answer.strip()))

In [ ]:
class MultiplyResponse(BaseModel):
    c_a_plus_b: int


multiply_in_prompt = (
    "Multiply {a_plus_b} by {c}. Your only output is the resulting number."
)

multiply_agent = LLMAgent[AddResponse, MultiplyResponse, State](
    name="multiply_agent",
    llm=LiteLLM(model_name="gpt-5.4-nano"),
    in_prompt=multiply_in_prompt,
    reset_transcript_on_run=True,
    stream_llm=True,
)


# Need a custom input content builder to make use of the global state
@multiply_agent.add_input_content_builder
def build_input_content_impl(in_args: AddResponse, *, exec_id: str) -> Content:
    ctx = multiply_agent.ctx
    return Content.from_formatted_prompt(
        multiply_agent.in_prompt, a_plus_b=in_args.a_plus_b, c=ctx.state.c
    )


@multiply_agent.add_output_parser
def parse_output_impl(
    final_answer: str, *, in_args: AddResponse | None = None, exec_id: str
) -> MultiplyResponse:
    return MultiplyResponse(c_a_plus_b=int(final_answer.strip()))

In [ ]:
state = State(b=3, c=6)
ctx = RunContext[State](state=state)

seq_agent = SequentialWorkflow[AddInputArgs, MultiplyResponse, State](
    name="seq_agent", subprocs=[add_agent, multiply_agent], ctx=ctx
)

In [ ]:
# Watch the workflow stream: both sub-agents run in sequence, the second
# consuming the first's output.
out = None
async for event in render_events(seq_agent.run_stream(in_args=AddInputArgs(a=2))):
    if isinstance(event, ProcPacketOutEvent):
        out = event.data

# Agents as tools

When agents are used as tools, their `in_args` become the tool inputs.

This is how one can implement a manager + helpers architecture.

In [ ]:
seq_tool = seq_agent.as_tool(
    tool_name="seq_agent_tool",
    tool_description=(
        "A sequential agent that adds 3 to a given integer, "
        "then multiplies the result by 5."
    ),
)

The JSON schema of `in_args` is preserved:

In [ ]:
seq_tool.in_type.model_json_schema()

In [ ]:
await seq_tool(a=15, ctx=ctx)

Stream from the tool

In [ ]:
state = State(b=3, c=6)
ctx = RunContext[State](state=state)

# ctx is passed to the sub-agents via the SequentialWorkflow
seq_agent = SequentialWorkflow[AddInputArgs, MultiplyResponse, State](
    name="seq_agent", subprocs=[add_agent, multiply_agent], ctx=ctx
)
seq_tool = seq_agent.as_tool(
    tool_name="seq_agent_tool",
    tool_description=(
        "A sequential agent that adds 3 to a given integer, "
        "then multiplies the result by 5."
    ),
)

async for event in render_events(seq_tool.run_stream(AddInputArgs(a=15), ctx=ctx)):
    pass

# Custom API providers and HTTP clients

In [ ]:
custom_provider = APIProvider(
    name="openrouter",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

http_client = httpx.AsyncClient(
    timeout=httpx.Timeout(10),
    limits=httpx.Limits(max_connections=10),
)

ctx = RunContext[None](printer=Printer())

chatbot = LLMAgent[None, list[int], None](
    name="chatbot",
    ctx=ctx,
    llm=OpenAILLM(
        model_name="google/gemma-4-31b-it",
        api_provider=custom_provider,
        http_client=http_client,
    ),
)


out = await chatbot.run(
    "Output a list of 3 integers from 0 to 10 as a python array, no talking",
)
print(out.payloads[0])